# Assignment 3

# 📘 Deep Learning Assignment

👥 **Group members:** _[Write your names here]_  
🗓️ **Deadline:** See Studium  

📤 Submit:
- Both `.ipynb` and `.pdf` versions
- Restart and run all cells before exporting
- Follow the filename format:  
'GroupNumber_Student1_Student2.ipynb'
  `42_EbbaBergman_AkshaiParakkalSreenivasan.ipynb`

## 🧭 Before You Start
> ⚠️ **Important:**  
> Before running this notebook, you **must shut down all other notebooks and kernels**.  
> Running multiple notebooks at once can cause memory issues and slowdowns — especially with image data.  
> Use the stop icon (■) in the Jupyter sidebar to shut down any active sessions.

>  **If this notebook doesn't run at any point:**  
  >1. Try rerunning the cell
  >2. If stuck, interrupt the cell
  >3. Restart the kernel and run all necessary cells again

## Data
There's a whole thesis written based on this dataset, available here: http://uu.diva-portal.org/smash/get/diva2:1650957/FULLTEXT01.pdf . The thesis then resulted in a paper: https://openaccess.thecvf.com/content_ICCV_2017_workshops/w1/html/Wieslander_Deep_Convolutional_Neural_ICCV_2017_paper.html


A quote from Weislander et al. regarding the dataset:

"The cell samples were collected at Södersjukhuset in Stockholm. The patients have mixed genders, are non smoking, some are human papillomavirus (HPV) positive and some are not, and they have an age span of 47-77 years. From each patient samples were collected with a brush that is scraped at areas of interest in the oral cavity. Each scrape is then smeared out on a glass, which is then stained to highlight important cellular structures"

## Notes
This assignment is inspired by Phil Harrisons lab for Pharmaceutical Bioinformatics and Sequence Analysis  from 2021

**You are not allowed to use the exact same network and configurations as in these texts, or copy directly from the labs, but you are allowed to use the same base network if you are using transfer learning. The network you use here must be significantly different from any other network you've used in the labs or read about in the paper above**  

You may discuss theory with other groups, but not code nor share code. You may not use ChatGPT4 or similar programs to generate a solution, code, nor answers to the questions

 <span style="color:red"> Solutions deemed to similar will be run through a program designed to detect coding plagirism and if flagged you will be reported for plagirism. </span> 
 
 One hand in per group.
 


 ## Hand in
 Hand in one pdf file and one .ipynb file. The notebook must be runnable with the data, any teacher might run the notebook to check that it actually works. Notebooks that do not run will need to be redone without further feedback. You may redo this hand in twice before failing the assignment all togehter. 
 
## Report structure
 You must follow the reportstructure to pass the assignment. Notebooks that do not follow the reportstructure will be sent back to be redone without further feedback.
 The full reporstructure is specified at the end of this notebook


## Goal
Design a neural network that achieves at least 82% balanced accuracy while demonstrating an understanding of the decisions made throughout the process as well as the strengths and limitations of the final model.

---

## Tasks

1. Build a neural network that achieves less than 65% balanced accuracy on the validation set

2. Build a neural network that achieves at least 82% balanced accuracy on the validation set, demonstrating the 

    * Start with an initial model, improve it step by step until you reach the target accuracy
        *
    * Show atleast 2 distinct stages of your model
        * For each stage, clearly describe:
            - what you changed
            - why you made that choice 
        * As you build on the previous model you need demonstrate the new improvements based on the lates model. I.e. if the inital model is A, after your first improvement your model is B, then the the next improvement will be made on B (or later), not A. 
        * If you show different stages further apart you need to show the previous stage as well. E.g A->B, D->E

        * If your  model reached the target accuracy with fewer than 2 changes:
            Continue modifying the model until you have at least two changes in total
            Reflect on whether later changes improved or worsened performance

3. Report your final balanced accuracy at the top of the notebook (in the question section)

4. Answer all questions in the question section

## Question section

### Accuracies

**Total balanced accuracy:**  
*(Report your final model performance here, for both the training, validation, and test sets)*

**Accuracy per class:**  
*(List the performance for each class and comment on any differences)*

**Any other measurements you would like to include:**  
*(For example: precision, recall, F1-score, confusion matrix, etc. Briefly explain why you chose these)*


### Questions

**Q1:** If a clinician requires 80% accuracy in their model, would you recommend using the model you have developed? Why or why not?  
(Think about reliability, generalisation, and whether accuracy alone is sufficient.)

**A1:**
*your answer here*


**Q2:** Your model went through several changes. Which specific change had the largest impact on performance, and why do you think that was the case?

**A2:**
*your answer here*

**Q3:** In this dataset, patients were mixed together before splitting the data into training and validation sets. What is the main drawback of this approach?

**A3:**
*your answer here*

**Q4:** What is a potential benefit of mixing patients before splitting the data?

**A4:**
*your answer here*

**Q5:** How might the results change if you instead split the data by patient (i.e. ensuring that cells from the same patient are only in either training or validation, not both)? Explain your reasoning.

**A1:**
*your answer here*

**Q6:** Besides accuracy, what other evaluation metrics could be important for this task, and why?

**A6:**
*your answer here*

**Q7:** If your model performs well on the validation set but poorly in a real clinical setting, what could be the reason for this mismatch?

**A7:**
*your answer here*

# Set up

In [ ]:
import cv2
import os
import random

import numpy as np
import pandas as pd
import IPython
import matplotlib.pyplot as plt


from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight

from datetime import datetime

from importlib.machinery import SourceFileLoader

from IPython.display import display, HTML

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator



In [ ]:
# Configure GPUs
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    # Currently, memory growth needs to be the same across GPUs
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
      print("Done setting memory_growth")
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except Exception as e:
    # Memory growth must be set before GPUs have been initialized
    print("If you get this , probably done already")
    # Catch the exception and display a custom HTML message with red text
    message = """There was some problems setting up the GPU,
                 it is probably best to restart kernel and clear
                 all outputs before starting over
              """
    display(HTML(f"<div style='color: red;'><strong>Warning:</strong>{message}</div>"))
    print(e)

# Paths

In [ ]:
current_directory = os.getcwd()
print(current_directory)

In [ ]:

base_path = '/home/jovyan/BigDataDL/TeacherVersions/'  # Students will need to update

assignments_directory = os.path.join(base_path, 'Assignments')

hpv_images_directory = os.path.join(base_path, 'Data', 'HPV_slides')  # Students will need to update

In [ ]:
from load_helpers import load_helpers
helpers, cnn_helper, plot_helper = load_helpers(assignments_directory)

### Setup HPV data

Extract the HPV data  using the HPV_v2_data_prep.ipynb notebook
You will need to update the directories in that notebook, and the directories below to match

In [ ]:
# directories for our training, validation and test splits
train_directory = os.path.join(hpv_images_directory, 'train')
validation_directory = os.path.join(hpv_images_directory, 'validation')
test_directory = os.path.join(hpv_images_directory, 'test')

# directory with our training healthy cell images
train_healthy_directory = os.path.join(train_directory, 'healthy')

# directory with our training tumor cell images
train_tumor_directory = os.path.join(train_directory, 'tumor')

# directory with our validation healthy cell images
validation_healthy_directory = os.path.join(validation_directory, 'healthy')

# directory with our validation tumor cell images
validation_tumor_directory = os.path.join(validation_directory, 'tumor')

# directory with our test healthy cell images
test_healthy_directory = os.path.join(test_directory, 'healthy')

# directory with our test tumor cell images
test_tumor_directory = os.path.join(test_directory, 'tumor')

In [ ]:
#### Data check

print('Number of  healthy images for training:', len(os.listdir(train_healthy_directory)))
print('Number of  tumor images for training:', len(os.listdir(train_tumor_directory)))
print('')
print('Number of  healthy images for validation:', len(os.listdir(validation_healthy_directory)))
print('Number of  tumor imagess for validation:', len(os.listdir(validation_tumor_directory)))
print('')
print('Number of  healthy images for testing:', len(os.listdir(test_healthy_directory)))
print('Number of  tumor images for testing:', len(os.listdir(test_tumor_directory)))

> ⚠️ **Important:**  
> **shut down the HPV_v3_data_prep.ipynb notebook**.  
> Running multiple notebooks at once can cause memory issues and slowdowns — especially with image data.  
> Use the stop icon (■) in the Jupyter sidebar to shut down any active sessions.

## Plot sample images 

In [ ]:
plot_helper.show_random_images_hpv_specific(train_healthy_directory)

## Other setup

In [ ]:
x_len = 96
y_len = 96
batch_size = 32
n_epochs = 10
class_labels=['healthy', 'tumor']

## Note: You might want to change the datagenerators, if so include that in yours steps below,
# ** do not add the changes here unless they're needed for your first model**
# Changes done to make another architecture run will not count as one of the 3 changes you need to make.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=10,
    zoom_range=0.1
)

validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

print('TRAINING DATA:')
train_generator = train_datagen.flow_from_directory(
    train_directory,
    target_size=(x_len, y_len),
    batch_size=batch_size,
    color_mode='grayscale',
    class_mode='binary'
)

print('')
print('VALIDATION DATA:')
validation_generator = validation_datagen.flow_from_directory(
    validation_directory,
    target_size=(x_len, y_len),
    batch_size=batch_size,
    color_mode='grayscale',
    class_mode='binary',
    shuffle=False
)

print('')
print('TEST DATA:')
test_generator = test_datagen.flow_from_directory(
    test_directory,
    target_size=(x_len, y_len),
    batch_size=batch_size,
    color_mode='grayscale',
    class_mode='binary',
    shuffle=False
)

class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights = dict(zip(np.unique(train_generator.classes), class_weights_array))

# Functions

In [ ]:
  
def get_image_data_flat_from_file(data_directory, image_paths):
    file_names = image_paths.values.flatten() # Assumes image_paths come in df[image_path_column_name] structure due to lab
    image_data = np.array([np.array(cv2.imread(data_directory + file_name)) for file_name in file_names])
    flattened_image_data = image_data.reshape(image_data.shape[0], -1)
    return flattened_image_data


# STUDENT CODE BEGINS

You can add new markdown and code cells in the menues at the top of the notebook interface.

# 🧾 Report Structure

For each task, use the following structure:

## General structure
- Use **Markdown cells** to structure and explain your work  
- Use **Code cells** for models, training, and evaluation  
- The **explanation must come before the code** for each model or improvement  
- Avoid vague statements such as “it improved performance” — **quantify your results where possible**

---

## Per task

- Markdown cell: **Task number** as a header
- Markdown cell: **Description of the task**
    - Describe:
        - choices made  
        - motivations (for *all* models, not only changes)  
        - reflections on results and any changes made (where appropriate)  
    - See example below for how to structure this

- Markdown cell: **Start of code section**
- Code cells:
    - Model definition  
    - Training  
    - Evaluation  

- Use **Markdown cells** to clearly separate different parts of your code

---

## For each model, include:

- Motivation for the model design (even if it is not a change from a previous model)  
- Model code  
- Comment indicating what part of the model has changed (where appropriate)  
- Validation curves  
- Confusion matrix  
- Accuracy and balanced accuracy  

  
---

### 🔄 Example: Switching to ELU Activation  
*(Note: You cannot use ELU activation as one of your improvements, nor copy any of this language directly. Plagiarism will be reported.)*

#### Structure of answer

**1. Observation:**  
Training and validation accuracy increased only slowly for cnn_model_1, and there were signs of flat gradients in the first few epochs.

**2. Interpretation:**  
This suggested that the ReLU activations may have led to dead neurons early in training, especially since some of the inputs may have low or negative-valued distributions after normalisation.

**3. Change Made:**  
We replaced ReLU with ELU (Exponential Linear Unit) in all convolutional and dense layers.

**4. Justification:**  
Unlike ReLU, which outputs zero for all negative inputs, ELU allows small negative values. This helps avoid “dead” neurons and encourages more consistent gradient flow during early training. ELU can improve convergence in deeper networks or datasets where feature values include negative ranges.

**5. Short Discussion of Results:**  
With ELU, the training converged slightly faster, and validation accuracy increased more smoothly. This supports the idea that ELU helped mitigate early learning stagnation. It was a worthwhile change for this setting.

---

#### Answer as text

For cnn_model_1 we observed that both training and validation accuracy improved only gradually during the early epochs. This suggested that some neurons might have become inactive due to the use of ReLU activation. Since ReLU outputs zero for all negative inputs, it can sometimes cause "dead neurons," especially when inputs are close to zero after normalisation.

To address this, we replaced ReLU with ELU (Exponential Linear Unit) in all convolutional and dense layers. ELU allows small negative outputs, which helps preserve gradient flow and prevents early layers from becoming inactive. This change led to slightly faster convergence and a smoother increase in validation accuracy.

The improvement supports the idea that ELU helped reduce early learning stagnation, and it was a suitable choice for this setting.